# Investment Data EDA (RW-KE-ET-SS 2021-2025)

This notebook performs detailed exploratory data analysis on the investment and loan portfolio dataset. It includes:

- Schema and data quality audit
- Missingness and duplicate diagnostics
- Numeric distributions and outlier checks
- Repayment and delinquency analysis
- Temporal trends
- Geographic and demographic segmentation
- Anomaly and consistency checks

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
np.random.seed(42)

In [ ]:
def find_ml_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / 'Anomynized data').exists() and (p / 'notebooks').exists():
            return p
        if (p / 'ml' / 'Anomynized data').exists():
            return p / 'ml'
    raise FileNotFoundError('Could not locate ml root containing Anomynized data directory.')

def load_csv(csv_path: Path) -> pd.DataFrame:
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(csv_path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(csv_path)

ml_root = find_ml_root()
data_dir = ml_root / 'Anomynized data' / 'Anomynized data' / 'Anomynized data'
csv_path = data_dir / 'Investment data_all clients_RW-KE-ET-SS_2021-2025_Inkomoko.csv'

df = load_csv(csv_path)
print(f'ML root: {ml_root}')
print(f'CSV path: {csv_path}')
print(f'Shape: {df.shape}')

In [ ]:
date_candidates = [c for c in df.columns if 'date' in c.lower() or 'year' in c.lower()]
for c in date_candidates:
    if 'year' in c.lower():
        df[c] = pd.to_numeric(df[c], errors='coerce')
    else:
        df[c] = pd.to_datetime(df[c], errors='coerce', dayfirst=False)

numeric_hint = ('amount', 'paid', 'balance', 'due', 'fee', 'principal', 'interest', 'applied', 'approved', 'disbursed', 'scheduled', 'age', 'cycle', 'duration')
for c in df.columns:
    if any(k in c.lower() for k in numeric_hint):
        df[c] = (
            df[c]
            .astype(str)
            .str.replace(',', '', regex=False)
            .str.replace('$', '', regex=False)
            .str.replace('#REF!', '', regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors='ignore')

schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'null_count': [int(df[c].isna().sum()) for c in df.columns],
    'null_pct': [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns]
}).sort_values(['null_pct', 'n_unique'], ascending=[False, True])

display(schema.head(30))
display(df.head())

In [ ]:
print('Duplicate row count:', int(df.duplicated().sum()))
if 'loanNumber' in df.columns:
    print('Duplicate loanNumber count:', int(df['loanNumber'].duplicated().sum()))

missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
display((missing * 100).round(2).rename('missing_pct').to_frame().head(30))

if not missing.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing.head(25).sort_values().plot(kind='barh', ax=ax, color='#e07a5f')
    ax.set_title('Top 25 Columns by Missing Percentage')
    ax.set_xlabel('Missing ratio')
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
display(df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.head(30))

money_cols = [c for c in numeric_cols if any(k in c.lower() for k in ['amount', 'paid', 'balance', 'due', 'fee', 'principal', 'interest'])]
for c in money_cols[:12]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df[c].dropna().plot(kind='hist', bins=40, ax=axes[0], color='#3d405b')
    axes[0].set_title(f'Histogram: {c}')
    axes[0].set_xlabel(c)
    df[c].dropna().plot(kind='box', ax=axes[1], color='#81b29a')
    axes[1].set_title(f'Boxplot: {c}')
    plt.tight_layout()
    plt.show()

def iqr_outlier_ratio(s: pd.Series) -> float:
    s = s.dropna()
    if len(s) < 10:
        return np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    mask = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    return round(mask.mean() * 100, 2)

outlier_summary = pd.DataFrame({
    'column': money_cols,
    'outlier_pct_iqr': [iqr_outlier_ratio(df[c]) for c in money_cols]
}).sort_values('outlier_pct_iqr', ascending=False)
display(outlier_summary.head(20))

In [ ]:
cat_cols = [c for c in ['country', 'strata', 'gender', 'repaymentStatus', 'loanType', 'businessCategory', 'businessSubsector', 'province', 'district'] if c in df.columns]
for c in cat_cols:
    vc = df[c].fillna('MISSING').value_counts(dropna=False).head(20)
    display(vc.rename('count').to_frame())
    fig, ax = plt.subplots(figsize=(10, 4))
    vc.sort_values().plot(kind='barh', ax=ax, color='#f2cc8f')
    ax.set_title(f'Top categories: {c}')
    ax.set_xlabel('count')
    plt.tight_layout()
    plt.show()

if {'repaymentStatus', 'country'}.issubset(df.columns):
    ctab = pd.crosstab(df['country'], df['repaymentStatus'], normalize='index')
    display((ctab * 100).round(2))
    fig, ax = plt.subplots(figsize=(10, 5))
    ctab.plot(kind='bar', stacked=True, ax=ax, colormap='tab20c')
    ax.set_title('Repayment Status Distribution by Country')
    ax.set_ylabel('ratio')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'disbursementDate' in df.columns:
    tmp = df.dropna(subset=['disbursementDate']).copy()
    tmp['disb_month'] = tmp['disbursementDate'].dt.to_period('M').astype(str)
    monthly_counts = tmp['disb_month'].value_counts().sort_index()
    monthly_amount = tmp.groupby('disb_month')['disbursedAmount'].sum(min_count=1) if 'disbursedAmount' in tmp.columns else None

    fig, ax = plt.subplots(figsize=(13, 4))
    monthly_counts.plot(ax=ax, color='#2a9d8f')
    ax.set_title('Monthly Loan Disbursement Count')
    ax.set_ylabel('count')
    ax.tick_params(axis='x', rotation=90)
    plt.tight_layout()
    plt.show()

    if monthly_amount is not None:
        fig, ax = plt.subplots(figsize=(13, 4))
        monthly_amount.plot(ax=ax, color='#264653')
        ax.set_title('Monthly Disbursed Amount (Total)')
        ax.set_ylabel('amount')
        ax.tick_params(axis='x', rotation=90)
        plt.tight_layout()
        plt.show()

if {'cycle', 'repaymentStatus'}.issubset(df.columns):
    cycle_perf = pd.crosstab(df['cycle'], df['repaymentStatus'], normalize='index')
    display((cycle_perf * 100).round(2))

In [ ]:
anomaly_report = {}

if 'loanNumber' in df.columns:
    anomaly_report['missing_loanNumber'] = int(df['loanNumber'].isna().sum())

for c in [col for col in df.columns if any(k in col.lower() for k in ['balance', 'due', 'paid', 'amount']) and str(df[col].dtype) != 'object']:
    anomaly_report[f'negative_{c}'] = int((df[c] < 0).sum())

if {'approvedAmount', 'disbursedAmount'}.issubset(df.columns):
    anomaly_report['disbursed_gt_approved'] = int((df['disbursedAmount'] > df['approvedAmount']).fillna(False).sum())

if {'appliedAmount', 'approvedAmount'}.issubset(df.columns):
    anomaly_report['approved_gt_applied'] = int((df['approvedAmount'] > df['appliedAmount']).fillna(False).sum())

display(pd.Series(anomaly_report, name='count').sort_values(ascending=False).to_frame())

print('Key Findings (auto-generated prompts):')
print('- Focus on columns with high missingness before modeling.')
print('- Validate whether negative monetary values are valid accounting events or data errors.')
print('- Investigate segments with elevated overdue amounts and closed written-off statuses.')
print('- Review country/province-level repayment heterogeneity for risk strategies.')